In [2]:
import fitz  # PyMuPDF for PDFs
import os
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Download NLTK resources (first time only)
nltk.download("stopwords")
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shiva\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shiva\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shiva\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shiva\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
# Step 2 – Extract Text from Files

# We need functions to handle both PDF and Word:

def extract_text_from_pdf(pdf_path):
    """Extract all text from a PDF file."""
    text = ""
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text()
    return text

def extract_text_from_word(docx_path):
    """Extract all text from a Word (.docx) file."""
    doc = docx.Document(docx_path)
    return "\n".join([para.text for para in doc.paragraphs])

In [11]:
# Step 3 – Preprocessing

# Clean text (lowercase, remove stopwords, punctuation, lemmatize):

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)  # remove punctuation
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and w.isalpha()]
    return tokens

In [12]:
import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

job_description = """
We are looking for a Full-Stack Engineer who understands Node.js, React, TypeScript, AWS, Docker,
GraphQL, PostgreSQL, DynamoDB, Jest, Playwright, CI/CD, Webpack, Vite.
"""

# Process text
doc = nlp(job_description)

# Extract candidate skills: proper nouns, nouns, and abbreviations
skills = set()
for token in doc:
    # Keep proper nouns, nouns, and uppercase abbreviations
    if token.pos_ in ["PROPN", "NOUN"] or token.text.isupper():
        skills.add(token.text)

print("Extracted Skills:", skills)

In [13]:
def calculate_score(cv_skills, jd_skills):
    if not jd_skills:
        return 0, set()
    match_count = len(cv_skills.intersection(jd_skills))
    score = (match_count / len(jd_skills)) * 100
    gap = jd_skills - cv_skills
    return score, gap

In [14]:
def process_job_description(jd_input, is_file=True):
    """Process Job Description (text or file)."""
    if is_file:
        if jd_input.endswith(".pdf"):
            jd_text = extract_text_from_pdf(jd_input)
        elif jd_input.endswith(".docx"):
            jd_text = extract_text_from_word(jd_input)
        else:
            raise ValueError("Unsupported JD format. Use PDF or DOCX.")
    else:
        jd_text = jd_input  # directly provided text
    
    jd_tokens = preprocess(jd_text)
    return extract_skills(jd_tokens, SKILL_SET)

def rank_cvs(cv_folder, jd_skills):
    results = []
    for cv_file in os.listdir(cv_folder):
        if cv_file.endswith(".pdf"):
            cv_text = extract_text_from_pdf(os.path.join(cv_folder, cv_file))
        elif cv_file.endswith(".docx"):
            cv_text = extract_text_from_word(os.path.join(cv_folder, cv_file))
        else:
            continue
        
        cv_tokens = preprocess(cv_text)
        cv_skills = extract_skills(cv_tokens, SKILL_SET)
        score, gap = calculate_score(cv_skills, jd_skills)
        
        results.append({
            "Candidate": cv_file,
            "Score": round(score, 2),
            "Matched Skills": list(cv_skills.intersection(jd_skills)),
            "Missing Skills": list(gap)
        })
    
    return sorted(results, key=lambda x: x["Score"], reverse=True)

In [19]:
if __name__ == "__main__":
    # OPTION 1: Job Description as text
    jd_text = """
    We are looking for a Data Scientist with skills in Python, SQL, Machine Learning, AWS, and strong communication.
    """
    jd_skills = process_job_description(jd_text, is_file=False)

    # OPTION 2: Job Description as a file
    # jd_skills = process_job_description(r"C:\Users\shiva\Desktop\Assignment\Sem - 4\S225 PRT604 PROFESSIONAL EXPERIENCE\job_description.pdf", is_file=True)

    # Folder containing CVs
    cv_folder = r"C:\Users\shiva\Desktop\Assignment\Sem - 4\S225 PRT604 PROFESSIONAL EXPERIENCE\CV"

    results = rank_cvs(cv_folder, jd_skills)

    for r in results:
        print(f"Candidate: {r['Candidate']} - Score: {r['Score']}%")
        print(f"Matched Skills: {r['Matched Skills']}")
        print(f"Missing Skills: {r['Missing Skills']}")
        print("-" * 60)

Candidate: Shiva_Giri_cv.pdf - Score: 50.0%
Matched Skills: ['communication', 'sql', 'python']
Missing Skills: ['machine', 'learning', 'aws']
------------------------------------------------------------


In [21]:
pip install python-docx


   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 3.4 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 2.6 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 2.8 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/4.0 MB 3.2 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.0 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.6 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   ---------------------------------------- 2/2 [python-docx]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import re
import string
import fitz  # PyMuPDF
import docx
import spacy
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

# ------------------------------
# Step 0 – NLTK Resources
# ------------------------------
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")

# ------------------------------
# Step 1 – Preprocessing Setup
# ------------------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w.isalpha() and w not in stop_words]
    return tokens

# ------------------------------
# Step 2 – Text Extraction
# ------------------------------
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join([page.get_text() for page in doc])

def extract_text_from_word(docx_path):
    doc = docx.Document(docx_path)
    return "\n".join([para.text for para in doc.paragraphs])

# ------------------------------
# Step 3 – Extract Multi-word Skills from JD
# ------------------------------
nlp = spacy.load("en_core_web_sm")

def extract_skills_from_jd(jd_text):
    doc = nlp(jd_text)
    skills = set()

    # Noun chunks for multi-word skills
    for chunk in doc.noun_chunks:
        skill = chunk.text.lower().strip()
        if skill:
            skills.add(skill)

    # Proper nouns and uppercase abbreviations
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"] or token.text.isupper():
            skills.add(token.text.lower())

    # Capture punctuated tech skills like CI/CD, Node.js, C++
    custom_skills = re.findall(r"\b[a-zA-Z0-9./+-]+(?:/[a-zA-Z0-9./+-]+)*\b", jd_text.lower())
    for s in custom_skills:
        skills.add(s)

    return skills

# ------------------------------
# Step 4 – Match CV Skills
# ------------------------------
def extract_skills_from_cv(cv_text, jd_skills):
    cv_text_lower = cv_text.lower()
    matched_skills = set()
    for skill in jd_skills:
        if skill in cv_text_lower:
            matched_skills.add(skill)
    return matched_skills

def calculate_score(cv_skills, jd_skills):
    if not jd_skills:
        return 0, set()
    match_count = len(cv_skills.intersection(jd_skills))
    score = (match_count / len(jd_skills)) * 100
    gap = jd_skills - cv_skills
    return score, gap

# ------------------------------
# Step 5 – Rank CVs
# ------------------------------
def rank_cvs(cv_folder, jd_skills):
    results = []
    for cv_file in os.listdir(cv_folder):
        cv_path = os.path.join(cv_folder, cv_file)
        if cv_file.endswith(".pdf"):
            cv_text = extract_text_from_pdf(cv_path)
        elif cv_file.endswith(".docx"):
            cv_text = extract_text_from_word(cv_path)
        else:
            continue

        cv_skills = extract_skills_from_cv(cv_text, jd_skills)
        score, gap = calculate_score(cv_skills, jd_skills)
        results.append({
            "Candidate": cv_file,
            "Score": round(score, 2),
            "Matched Skills": ", ".join(cv_skills),
            "Missing Skills": ", ".join(gap),
            "JD Skills": ", ".join(jd_skills)
        })

    return sorted(results, key=lambda x: x["Score"], reverse=True)

# ------------------------------
# Step 6 – Read JD from Text File
# ------------------------------
def read_jd_from_file(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Job description file not found: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        jd_text = f.read()
    return jd_text

# ------------------------------
# Step 7 – Save Excel with Highlights and Summary
# ------------------------------
def save_excel_with_summary(df, save_folder):
    output_excel = os.path.join(save_folder, "cv_ranking_summary.xlsx")
    output_csv = os.path.join(save_folder, "cv_ranking.csv")

    wb = Workbook()
    ws_data = wb.active
    ws_data.title = "CV Ranking"

    # Write data
    for r in dataframe_to_rows(df, index=False, header=True):
        ws_data.append(r)

    # Highlight missing skills in red
    red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    missing_col = None
    for idx, cell in enumerate(ws_data[1], 1):
        if cell.value == "Missing Skills":
            missing_col = idx
            break
    if missing_col:
        for row in range(2, ws_data.max_row + 1):
            cell = ws_data.cell(row=row, column=missing_col)
            if cell.value and cell.value.strip():
                cell.fill = red_fill

    # Summary sheet
    ws_summary = wb.create_sheet(title="Summary")
    total_candidates = len(df)
    average_score = round(df["Score"].mean(), 2)
    top_candidates = df.sort_values(by="Score", ascending=False).head(3)

    ws_summary.append(["Metric", "Value"])
    ws_summary.append(["Total Candidates", total_candidates])
    ws_summary.append(["Average Score", average_score])
    ws_summary.append([])  # Empty row
    ws_summary.append(["Top 3 Candidates", "Score"])
    for _, row in top_candidates.iterrows():
        ws_summary.append([row["Candidate"], row["Score"]])

    wb.save(output_excel)
    df.to_csv(output_csv, index=False)

    print(f"Excel saved with summary and highlights: {output_excel}")
    print(f"CSV saved: {output_csv}")

# ------------------------------
# Step 8 – Main Execution
# ------------------------------
if __name__ == "__main__":
    # Job description file
    jd_file = r"C:\Users\shiva\Desktop\Assignment\Sem - 4\S225 PRT604 PROFESSIONAL EXPERIENCE\CV\job_description.txt"
    jd_text = read_jd_from_file(jd_file)
    jd_skills = extract_skills_from_jd(jd_text)

    # CV folder
    cv_folder = r"C:\Users\shiva\Desktop\Assignment\Sem - 4\S225 PRT604 PROFESSIONAL EXPERIENCE\CV"

    results = rank_cvs(cv_folder, jd_skills)
    df = pd.DataFrame(results)

    # Display results
    for r in results:
        print(f"Candidate: {r['Candidate']} - Score: {r['Score']}%")
        print(f"Matched Skills: {r['Matched Skills']}")
        print(f"Missing Skills: {r['Missing Skills']}")
        print("-" * 60)

    # Save Excel and CSV in CV folder
    save_excel_with_summary(df, cv_folder)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject